# Lesson 01 - Pengenalan kepada Ejen AI

Selamat datang ke pelajaran pertama dalam kursus **Ejen AI untuk Pemula**!

Sebuah **ejen AI** adalah program yang menggunakan model bahasa besar (LLM) sebagai enjin pemikirannya dan boleh mengambil *tindakan* dalam dunia nyata — memanggil API, membuat pertanyaan pangkalan data, atau menjalankan kod — untuk mencapai matlamat bagi pihak pengguna.

Dalam buku nota ini anda akan membina ejen pertama anda: sebuah **Ejen Pelancongan** yang mencadangkan destinasi percutian. Sepanjang jalan anda akan belajar bagaimana untuk:

1. Sambungkan ke Perkhidmatan Azure AI Foundry Agent menggunakan **Rangka Kerja Ejen Microsoft**.
2. Berikan ejen sebuah **alat** — fungsi Python biasa yang boleh dipanggilnya.
3. Jalankan ejen dan periksa maklum balasnya.
4. Alirkan maklum balas ejen token demi token.


## Persediaan

Sebelum menjalankan notebook ini, pastikan anda mempunyai:

1. **Projek Azure AI Foundry** dengan model sembang yang telah ditempatkan (contohnya `gpt-4o-mini`).
2. **Log masuk dengan Azure CLI** — jalankan `az login` dalam terminal anda.
3. **Tetapkan pembolehubah persekitaran yang diperlukan:**
   - `AZURE_AI_PROJECT_ENDPOINT` — titik hujung projek Azure AI Foundry anda.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nama model yang anda tempatkan.

Sel selanjutnya memasang pakej Python yang anda perlukan.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Membuat Ejen Pertama Anda

Sebuah ejen memerlukan dua perkara:

- **Arahan** yang memberitahunya *siapa dia* dan *cara berkelakuan* (prompt sistem).
- **Alat** — fungsi Python yang dihias dengan `@tool` yang boleh dipanggil oleh ejen untuk mendapatkan maklumat atau melaksanakan tindakan.

Di bawah kami mendefinisikan alat ringkas yang mengembalikan senarai destinasi percutian popular. Ejen akan menggunakan alat ini apabila pengguna meminta cadangan perjalanan.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Untuk pengalaman yang lebih interaktif anda boleh **stream** respons ejen. Daripada menunggu balasan penuh, ejen akan mengeluarkan kepingan teks semasa ia dijana. Ini amat berguna dalam antara muka sembang di mana anda ingin memaparkan output secara masa nyata.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Ringkasan

Dalam pelajaran ini anda telah belajar bagaimana untuk:

- **Mencipta penyedia** yang berhubung dengan Azure AI Foundry Agent Service melalui `AzureAIProjectAgentProvider`.
- **Menentukan alat** menggunakan hiasan `@tool` supaya agen dapat memanggil fungsi Python anda.
- **Jalankan agen** dengan mesej pengguna dan cetak balasannya.
- **Alirkan respons** untuk output masa nyata.

Dalam pelajaran seterusnya kami akan meneroka rangka kerja agen dengan lebih mendalam dan belajar bagaimana memberi agen alat yang lebih berkuasa dan kebolehan penalaran berbilang langkah.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
